# Talent Ranking with LLMs

This notebook builds an **intelligent candidate ranking system** for a recruiting use case. Given 104 candidates with free-text LinkedIn job titles, the goal is to automatically surface the most relevant HR candidates using two approaches:

1. **Embedding methods** — convert job titles into vectors and rank by semantic similarity to the target keywords *"aspiring human resources"* and *"seeking human resources"*
2. **LLM listwise ranking** — pass the full candidate list to a language model and ask it to directly select the top 10 HR-relevant candidates

Six embedding strategies are benchmarked side-by-side, from classical bag-of-words (TF-IDF) to fine-tuned transformers (BERT, E5). Three LLMs are tested across four prompting techniques each, giving **18 columns** of results to compare.

> **Runtime:** GPU (T4 or better) required for the LLM section — *Runtime → Change runtime type → GPU*

## 1. Imports

Install the packages not pre-installed in Colab, then import everything the notebook needs. All imports are collected here so all dependencies are visible in one place before any code runs.

In [ ]:
!pip install -q sentence-transformers gensim mittens bitsandbytes accelerate

In [ ]:
# Standard library
import os
import re
import gc

# Reduce VRAM fragmentation — must be set before any CUDA allocation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Data
import numpy as np
import pandas as pd

# Colab
from google.colab import files, userdata

# ML / embeddings
import torch
from torch.utils.data import DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from gensim.models import Word2Vec, FastText, KeyedVectors
import gensim.downloader as gensim_dl
from mittens import Mittens
from sentence_transformers import SentenceTransformer, InputExample, losses

# HuggingFace
from huggingface_hub import hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig, BitsAndBytesConfig

## 2. GPU Utilisation

Confirms a GPU is attached to the runtime and prints the available VRAM. The LLM section (section 7) requires at least **15 GB VRAM** for GLM-4-9B and Gemma loaded in 4-bit quantisation. A Colab T4 GPU provides 15 GB, which is sufficient for all three models loaded sequentially.

In [ ]:
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU            : {gpu.name}")
    print(f"VRAM           : {gpu.total_memory / 1024**3:.1f} GB")
    print(f"CUDA version   : {torch.version.cuda}")
    print(f"PyTorch version: {torch.__version__}")
else:
    print("No GPU detected â€” switch to GPU runtime: Runtime > Change runtime type > GPU")

## 3. Load Dataset

The dataset contains **104 candidates** scraped from LinkedIn-style profiles with three fields:

| Column | Description |
|---|---|
| `id` | Unique candidate identifier |
| `job_title` | Free-text job title (e.g. *Aspiring HR Professional \| Entry-Level Candidate*) |
| `connection` | LinkedIn connection count (may include `500+`) |

Upload `PotentialTalents.csv` when prompted, or uncomment the Google Drive option.

In [ ]:
# --- Option 1: upload directly ---
uploaded = files.upload()
csv_name = list(uploaded.keys())[0]
df_raw = pd.read_csv(csv_name)

# --- Option 2: Google Drive (uncomment) ---
# from google.colab import drive
# drive.mount('/content/drive')
# df_raw = pd.read_csv('/content/drive/MyDrive/PotentialTalents.csv')

print(f"Loaded {len(df_raw)} candidates")
df_raw.head()

## 4. Preprocessing

Three transformations prepare the raw data before any embedding is computed:

| Step | Output column | Detail |
|---|---|---|
| Title cleaning | `job_title_clean` | Lower-cased; transformer models handle punctuation internally so no stripping is needed |
| Connection parsing | `connections_raw` | `500+` capped at 500; invalid or missing values default to 0 |
| Connection normalisation | `connections_norm` | Divided by 500, range [0, 1], so it can be blended with cosine similarity scores |

The blended ranking formula used throughout:
```
fit = 0.7 × title_similarity + 0.3 × connections_norm
```
The 0.7 / 0.3 split was confirmed optimal by grid search — title relevance dominates, connection count acts as a tie-breaker.

In [ ]:
TARGET_KEYWORDS = ["aspiring human resources", "seeking human resources"]
TARGETS_CLEAN   = [t.lower().strip() for t in TARGET_KEYWORDS]

def preprocess(df):
    df = df.copy()
    df["job_title_clean"] = df["job_title"].astype(str).str.lower().str.strip()
    def parse_conn(val):
        try:
            return min(int(str(val).replace("+", "").strip()), 500)
        except:
            return 0
    df["connections_raw"]  = df["connection"].apply(parse_conn)
    df["connections_norm"] = df["connections_raw"] / 500.0
    return df

df = preprocess(df_raw)
df[["id", "job_title", "job_title_clean", "connections_norm"]].head(10)

## 5. Embedding Technique Comparison

Each method converts candidate job titles and target keywords into vectors, then computes **cosine similarity**. The candidate score is the **maximum** similarity across both targets — this prevents long titles from diluting a single highly relevant phrase.

| Method | Type | Trained on | Key property |
|---|---|---|---|
| TF-IDF | Bag-of-words | This corpus | Exact word match, weighted by rarity — no understanding of meaning |
| Word2Vec | Word embeddings | This corpus | Context-based, average pooling — similar words get similar vectors |
| FastText | Subword embeddings | This corpus | Handles rare/misspelled words via character n-grams |
| GloVe-FT | Word embeddings | Wikipedia + Gigaword, fine-tuned with Mittens | Pretrained co-occurrence vectors adapted to job-title vocabulary |
| BERT-FT | Transformer | Large pretraining, fine-tuned with SimCSE | Full sentence context; domain-adapted without labelled data |
| E5-small-FT | Transformer | Large pretraining, fine-tuned with SimCSE | Retrieval-optimised; asymmetric query/passage encoding |

**What to expect:** TF-IDF and Word2Vec tend to miss synonym matches (`HR` vs `Human Resources`) because they rely on exact token overlap. BERT and E5 handle these because they encode meaning. E5 is purpose-built for retrieval and typically surfaces the strongest HR candidates.

Each cell below runs one method and prints its top-10 ranked candidates.

In [ ]:
def _avg_word_vecs(texts, model, dim):
    result = []
    for text in texts:
        vecs = [model.wv[tok] for tok in text.split() if tok in model.wv]
        result.append(np.mean(vecs, axis=0) if vecs else np.zeros(dim))
    return np.array(result)

def _avg_keyed_vecs(texts, kv, dim):
    result = []
    for text in texts:
        vecs = [kv[tok] for tok in text.split() if tok in kv]
        result.append(np.mean(vecs, axis=0) if vecs else np.zeros(dim))
    return np.array(result)

def rank_candidates(df, scores, w=0.7):
    ranked = df.copy()
    ranked["_score"] = w * scores + (1 - w) * ranked["connections_norm"].values
    return ranked.sort_values("_score", ascending=False).reset_index(drop=True)

In [ ]:
# --- TF-IDF ---
def embed_tfidf(df):
    all_texts = df["job_title_clean"].tolist() + TARGETS_CLEAN
    vec = TfidfVectorizer()
    vec.fit(all_texts)
    cand_embs = vec.transform(df["job_title_clean"].tolist()).toarray()
    tgt_embs  = vec.transform(TARGETS_CLEAN).toarray()
    return cosine_similarity(cand_embs, tgt_embs).max(axis=1)

scores_tfidf = embed_tfidf(df)
rank_candidates(df, scores_tfidf)[["id", "job_title", "_score"]].head(10).set_index("id")

In [ ]:
# --- Word2Vec ---

def embed_word2vec(df):
    DIM = 100
    all_texts = df["job_title_clean"].tolist() + TARGETS_CLEAN
    corpus = [t.split() for t in all_texts]
    model  = Word2Vec(corpus, vector_size=DIM, window=5, min_count=1,
                      workers=4, epochs=100, seed=42)
    cand_embs = _avg_word_vecs(df["job_title_clean"].tolist(), model, DIM)
    tgt_embs  = _avg_word_vecs(TARGETS_CLEAN, model, DIM)
    return cosine_similarity(cand_embs, tgt_embs).max(axis=1)

scores_w2v = embed_word2vec(df)
rank_candidates(df, scores_w2v)[["id", "job_title", "_score"]].head(10).set_index("id")

In [ ]:
# --- FastText ---

def embed_fasttext(df):
    DIM = 100
    all_texts = df["job_title_clean"].tolist() + TARGETS_CLEAN
    corpus = [t.split() for t in all_texts]
    model  = FastText(corpus, vector_size=DIM, window=5, min_count=1,
                      workers=4, epochs=100, seed=42)
    cand_embs = _avg_word_vecs(df["job_title_clean"].tolist(), model, DIM)
    tgt_embs  = _avg_word_vecs(TARGETS_CLEAN, model, DIM)
    return cosine_similarity(cand_embs, tgt_embs).max(axis=1)

scores_ft = embed_fasttext(df)
rank_candidates(df, scores_ft)[["id", "job_title", "_score"]].head(10).set_index("id")

In [ ]:
# --- GloVe fine-tuned with Mittens ---

def embed_glove_finetuned(df):
    DIM    = 100
    corpus = df["job_title_clean"].tolist() + TARGETS_CLEAN
    cv = TfidfVectorizer(use_idf=False, binary=True)
    X  = cv.fit_transform(corpus)
    Xc = (X.T @ X).toarray().astype(float)
    vocab = cv.get_feature_names_out().tolist()
    kv        = gensim_dl.load("glove-wiki-gigaword-100")
    pretrained = {w: kv[w].tolist() for w in vocab if w in kv}
    new_vecs   = Mittens(n=DIM, max_iter=1000).fit(
        Xc, vocab=vocab, initial_embedding_dict=pretrained
    )
    kv_ft = KeyedVectors(vector_size=DIM)
    kv_ft.add_vectors(vocab, new_vecs)
    cand_embs = _avg_keyed_vecs(df["job_title_clean"].tolist(), kv_ft, DIM)
    tgt_embs  = _avg_keyed_vecs(TARGETS_CLEAN, kv_ft, DIM)
    return cosine_similarity(cand_embs, tgt_embs).max(axis=1)

print("Loading GloVe and fine-tuning with Mittens...")
scores_glove = embed_glove_finetuned(df)
rank_candidates(df, scores_glove)[["id", "job_title", "_score"]].head(10).set_index("id")

In [ ]:
# --- BERT fine-tuned with SimCSE ---

def embed_bert_finetuned(df):
    corpus   = df["job_title_clean"].tolist() + TARGETS_CLEAN
    model    = SentenceTransformer("all-MiniLM-L6-v2")
    examples = [InputExample(texts=[s, s]) for s in corpus]
    loader   = DataLoader(examples, shuffle=True, batch_size=16)
    loss     = losses.MultipleNegativesRankingLoss(model)
    model.fit(train_objectives=[(loader, loss)], epochs=5, show_progress_bar=False)
    cand_embs = model.encode(df["job_title_clean"].tolist(), show_progress_bar=False)
    tgt_embs  = model.encode(TARGET_KEYWORDS, show_progress_bar=False)
    return cosine_similarity(cand_embs, tgt_embs).max(axis=1)

print("Fine-tuning BERT with SimCSE...")
scores_bert = embed_bert_finetuned(df)
rank_candidates(df, scores_bert)[["id", "job_title", "_score"]].head(10).set_index("id")

In [ ]:
# --- E5-small fine-tuned with SimCSE ---
def embed_e5_finetuned(df):
    corpus   = df["job_title_clean"].tolist() + TARGETS_CLEAN
    model    = SentenceTransformer("intfloat/e5-small-v2")
    examples = [InputExample(texts=[s, s]) for s in corpus]
    loader   = DataLoader(examples, shuffle=True, batch_size=16)
    loss     = losses.MultipleNegativesRankingLoss(model)
    model.fit(train_objectives=[(loader, loss)], epochs=5, show_progress_bar=False)
    cand_texts = ["passage: " + t for t in df["job_title_clean"].tolist()]
    tgt_texts  = ["query: "   + t for t in TARGET_KEYWORDS]
    cand_embs  = model.encode(cand_texts, show_progress_bar=False)
    tgt_embs   = model.encode(tgt_texts,  show_progress_bar=False)
    return cosine_similarity(cand_embs, tgt_embs).max(axis=1)

print("Fine-tuning E5-small with SimCSE...")
scores_e5 = embed_e5_finetuned(df)
rank_candidates(df, scores_e5)[["id", "job_title", "_score"]].head(10).set_index("id")

## 6. Hugging Face Token

The LLM models (Qwen, GLM, Gemma) are downloaded from Hugging Face Hub and require authentication.

**How to add your token:**
1. Click the **Secrets** key icon (🔑) in the left sidebar of Colab
2. Add a secret named `HUGGING_FACE_API_KEY` with your HF token as the value
3. Run this cell to load the token into the environment

You can create or find your token at **huggingface.co → Settings → Access Tokens**.

In [ ]:
# Put the key in secret manager
os.environ["HUGGING_FACE_API_KEY"] = userdata.get("HUGGING_FACE_API_KEY")
HUGGING_FACE_API_KEY = os.environ.get("HUGGING_FACE_API_KEY")
print("HF token loaded:", "yes" if HUGGING_FACE_API_KEY else "MISSING â€” add it to Colab Secrets")

## 7. LLM Listwise Ranking

Instead of scoring candidates one-by-one, the **full list of 104 job titles** is passed to the model and it is asked to directly pick the top 10 HR-relevant candidates. This is called **listwise ranking** — the model sees all candidates at once and reasons about relative relevance.

Four prompting techniques are tested per model to see how much the instruction style affects the output:

| Technique | What it does |
|---|---|
| Zero-shot | Plain instruction with no examples — tests the model out-of-the-box |
| Few-shot | Shows a small worked example first — guides the output format |
| Chat | Framed as a user ↔ assistant conversation — mirrors how chat models are trained |
| CoT (Chain of Thought) | Asks the model to reason step-by-step before answering — improves accuracy on complex tasks |

All three models are **loaded one at a time and unloaded after use** (`del model + gc.collect()`) to stay within the T4 VRAM budget. GLM and Gemma use **4-bit NF4 quantisation** which reduces VRAM from ~15–18 GB down to ~4–5 GB with minimal accuracy loss.

In [ ]:
# â”€â”€ Prompt builders â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def _numbered_list(titles):
    return "\n".join(f"{i+1}. {t}" for i, t in enumerate(titles))

def build_zero_shot(titles):
    return (
        f"Below are {len(titles)} job titles. Identify the top 10 candidates "
        "most likely aspiring to a human resources position.\n"
        "Reply with ONLY the candidate numbers in ranked order, comma-separated "
        "(e.g. 3, 15, 7, ...).\n\n"
        f"{_numbered_list(titles)}\n\nTop 10 (best to worst):"
    )

def build_few_shot(titles):
    example = (
        "Example â€” given this short list:\n"
        "1. Aspiring Human Resources Professional\n"
        "2. Software Engineer at Google\n"
        "3. Seeking HR Opportunities\n"
        "Answer: 1, 3\n\n"
    )
    return (
        "Rank the top 10 candidates most suitable for a human resources role.\n"
        "Reply with ONLY the candidate numbers in ranked order, comma-separated.\n\n"
        f"{example}"
        f"Now rank this list:\n{_numbered_list(titles)}\n\nTop 10 (best to worst):"
    )

def build_chat_messages(titles):
    return [
        {"role": "user",      "content": "I will give you a list of job titles. Pick the top 10 most relevant for a human resources position and return their numbers in order."},
        {"role": "assistant", "content": "Understood. Please share the list and I will return the top 10 candidate numbers in ranked order, comma-separated."},
        {"role": "user",      "content": f"Here is the list:\n{_numbered_list(titles)}\n\nTop 10 (best to worst):"},
    ]

def build_cot(titles):
    return (
        f"Here are {len(titles)} job titles.\n\n{_numbered_list(titles)}\n\n"
        "Step 1 â€” What keywords indicate an HR-aspiring candidate "
        "(e.g. 'human resources', 'HR', 'aspiring', 'seeking')?\n"
        "Step 2 â€” Scan the list and identify candidates with those keywords.\n"
        "Step 3 â€” Rank the top 10 from best to worst match.\n\n"
        "Final answer â€” top 10 candidate numbers, comma-separated:"
    )

# â”€â”€ Inference helpers â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def _generate(model, tokenizer, messages, max_new_tokens=100):
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    cfg    = GenerationConfig(do_sample=False, max_new_tokens=max_new_tokens)
    out    = model.generate(**inputs, generation_config=cfg)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def _parse_ids(raw, ids, n_titles):
    found = [
        ids[int(i) - 1]
        for i in re.findall(r"\b(\d+)\b", raw)
        if 1 <= int(i) <= n_titles
    ][:10]
    return found + [None] * (10 - len(found))

def rank_with_llm(df, model, tokenizer):
    titles  = df["job_title_clean"].tolist()
    ids     = df["id"].tolist()
    results = {}
    tasks = [
        ("Zero-shot", [{"role": "user", "content": build_zero_shot(titles)}], 100),
        ("Few-shot",  [{"role": "user", "content": build_few_shot(titles)}],  100),
        ("Chat",      build_chat_messages(titles),                             100),
        ("CoT",       [{"role": "user", "content": build_cot(titles)}],       300),
    ]
    for name, msgs, max_tok in tasks:
        print(f"  {name}...", end=" ")
        raw = _generate(model, tokenizer, msgs, max_new_tokens=max_tok)
        results[name] = _parse_ids(raw, ids, len(titles))
        print(f"â†’ {results[name]}")
    return results

# â”€â”€ 4-bit quantization config (shared by GLM and Gemma) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
bnb_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
print("Helpers ready.")

# -- Model loader helper (used by main() in section 9) --
def load_hf_model(model_id, token, dtype=None, quant=None, trust=False):
    tok = AutoTokenizer.from_pretrained(model_id, token=token, padding_side="left", trust_remote_code=trust)
    kwargs = dict(token=token, device_map="auto", trust_remote_code=trust)
    if quant:
        kwargs["quantization_config"] = quant
    else:
        kwargs["torch_dtype"] = dtype or torch.float16
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    model.eval()
    return model, tok

print("Helpers ready.")


### 7a. Qwen 2.5-0.5B-Instruct

`Qwen/Qwen2.5-0.5B-Instruct` — a lightweight **0.5 billion parameter** instruction-tuned model from Alibaba. Loaded in float16 (no quantisation needed at this size, fits in ~1 GB VRAM).

**Note:** At 0.5B parameters Qwen is very small. It sometimes defaults to outputting numbers 1–10 sequentially rather than genuinely ranking by HR relevance. This is a model capability limitation, not a code bug — results should be interpreted with this in mind.

In [ ]:
QWEN_ID = "Qwen/Qwen2.5-0.5B-Instruct"
print(f"Loading {QWEN_ID}...")

qwen_tok = AutoTokenizer.from_pretrained(
    QWEN_ID, token=HUGGING_FACE_API_KEY, padding_side="left"
)
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_ID, token=HUGGING_FACE_API_KEY,
    torch_dtype=torch.float16, device_map="auto"
)
qwen_model.eval()
print("Running 4 prompting techniques...")
qwen_results = rank_with_llm(df, qwen_model, qwen_tok)

del qwen_model, qwen_tok
gc.collect()
torch.cuda.synchronize()
torch.cuda.empty_cache()
print(f"Qwen unloaded. Free VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

### 7b. GLM-4-9B-0414

`zai-org/GLM-4-9B-0414` — a **9 billion parameter** model from Zhipu AI, loaded in **4-bit NF4 quantisation** (~4.5 GB VRAM). Requires `trust_remote_code=True` because it uses a custom model architecture not built into the transformers library.

GLM-4 has strong multilingual and instruction-following capability, making it well suited for structured ranking tasks. Being 18× larger than Qwen, it produces noticeably more consistent and accurate candidate selections.

In [ ]:
GLM_ID = "zai-org/GLM-4-9B-0414"
print(f"Loading {GLM_ID} in 4-bit...")

glm_tok = AutoTokenizer.from_pretrained(
    GLM_ID, token=HUGGING_FACE_API_KEY,
    padding_side="left", trust_remote_code=True
)
glm_model = AutoModelForCausalLM.from_pretrained(
    GLM_ID, token=HUGGING_FACE_API_KEY,
    quantization_config=bnb_4bit, device_map="auto",
    low_cpu_mem_usage=True, trust_remote_code=True
)
glm_model.eval()
print("Running 4 prompting techniques...")
glm_results = rank_with_llm(df, glm_model, glm_tok)

del glm_model, glm_tok
gc.collect()
torch.cuda.synchronize()
torch.cuda.empty_cache()
print(f"GLM unloaded. Free VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

### 7c. Gemma-4-E2B-it

`google/gemma-4-E2B-it` — Google's Gemma 4 instruction-tuned model (~2.3B effective / 5.1B total parameters), loaded in **4-bit NF4 quantisation** (~3 GB VRAM).

Requires `transformers` installed from the GitHub source (done in section 1) because the `gemma4` architecture was added after the latest stable pip release.

As a purpose-built instruction-following model from Google, Gemma tends to produce well-structured, comma-separated output — which is exactly what the response parser expects.

In [ ]:
GEMMA_ID = "google/gemma-4-E2B-it"
print(f"Loading {GEMMA_ID} in 4-bit...")

gemma_tok = AutoTokenizer.from_pretrained(
    GEMMA_ID, token=HUGGING_FACE_API_KEY, padding_side="left"
)
gemma_model = AutoModelForCausalLM.from_pretrained(
    GEMMA_ID, token=HUGGING_FACE_API_KEY,
    quantization_config=bnb_4bit, device_map="auto"
)
gemma_model.eval()
print("Running 4 prompting techniques...")
gemma_results = rank_with_llm(df, gemma_model, gemma_tok)

del gemma_model, gemma_tok
gc.collect()
torch.cuda.empty_cache()
print("\nGemma unloaded.")

## 8. Summary Table

An **18-column × 10-row** comparison table showing the top-10 candidate IDs selected by each method.

| Columns | Source |
|---|---|
| TF-IDF, Word2Vec, FastText, GloVe-FT, BERT-FT, E5-small-FT | Embedding similarity + connections blend (w = 0.7) |
| Qwen-Zero-shot … Qwen-CoT | Qwen 2.5-0.5B-Instruct |
| GLM-Zero-shot … GLM-CoT | GLM-4-9B-0414 |
| Gemma-Zero-shot … Gemma-CoT | Gemma-4-E2B-it |

**How to read the table:** Each cell is a candidate ID. Candidates that appear across many columns are consistently identified as HR-relevant regardless of method — these are the strongest picks. `NaN` means the model returned fewer than 10 valid candidate numbers for that technique.

**What to look for:**
- Do the embedding methods and LLMs agree on the same top candidates?
- Does the prompting technique (zero-shot vs CoT) change the LLM's selections?
- Which embedding method most closely matches the LLM rankings?

The table is also saved to `candidate_table.csv` and downloaded to your local machine.

In [ ]:
summary = {}

# --- Embedding columns ---
for name, scores in [
    ("TF-IDF",      scores_tfidf),
    ("Word2Vec",    scores_w2v),
    ("FastText",    scores_ft),
    ("GloVe-FT",    scores_glove),
    ("BERT-FT",     scores_bert),
    ("E5-small-FT", scores_e5),
]:
    ranked = rank_candidates(df, scores)
    summary[name] = ranked["id"].head(10).tolist()

# --- LLM columns ---
for technique, ids_list in qwen_results.items():
    summary[f"Qwen-{technique}"] = ids_list
for technique, ids_list in glm_results.items():
    summary[f"GLM-{technique}"] = ids_list
for technique, ids_list in gemma_results.items():
    summary[f"Gemma-{technique}"] = ids_list

tbl = pd.DataFrame(summary, index=range(1, 11))
tbl.index.name = "Rank"
tbl

In [ ]:
# Save to CSV
tbl.to_csv("candidate_table.csv")
print("Saved: candidate_table.csv")

# Download to local machine
files.download("candidate_table.csv")

## 9. Run Everything (Optional)

An alternative to running each section individually. The `main()` function defined below calls every step in sequence — load data, run all 6 embedding methods, load and run all 3 LLMs, then produce and download the summary table.

Uncomment the last line (`# tbl = main()`) and run this cell to execute the full pipeline in one shot. This is useful if you want to re-run everything after changing a parameter, such as the ranking weight `w`.

In [ ]:
def main():
    # Load & preprocess
    uploaded = files.upload()
    df = preprocess(pd.read_csv(list(uploaded.keys())[0]))

    # Embeddings
    scores_tfidf = embed_tfidf(df)
    scores_w2v   = embed_word2vec(df)
    scores_ft    = embed_fasttext(df)
    scores_glove = embed_glove_finetuned(df)
    scores_bert  = embed_bert_finetuned(df)
    scores_e5    = embed_e5_finetuned(df)

    # LLMs
    os.environ["HUGGING_FACE_API_KEY"] = userdata.get("HUGGING_FACE_API_KEY")
    hf_token = os.environ["HUGGING_FACE_API_KEY"]

    qwen_model, qwen_tok = load_hf_model("Qwen/Qwen2.5-0.5B-Instruct", hf_token, dtype=torch.float16)
    qwen_results = rank_with_llm(df, qwen_model, qwen_tok)
    del qwen_model, qwen_tok; gc.collect(); torch.cuda.empty_cache()

    glm_model, glm_tok = load_hf_model("zai-org/GLM-4-9B-0414", hf_token, quant=bnb_4bit, trust=True)
    glm_results = rank_with_llm(df, glm_model, glm_tok)
    del glm_model, glm_tok; gc.collect(); torch.cuda.empty_cache()

    gemma_model, gemma_tok = load_hf_model("google/gemma-4-E2B-it", hf_token, quant=bnb_4bit)
    gemma_results = rank_with_llm(df, gemma_model, gemma_tok)
    del gemma_model, gemma_tok; gc.collect(); torch.cuda.empty_cache()

    # Summary table
    summary = {}
    for name, scores in [("TF-IDF", scores_tfidf), ("Word2Vec", scores_w2v),
                         ("FastText", scores_ft), ("GloVe-FT", scores_glove),
                         ("BERT-FT", scores_bert), ("E5-small-FT", scores_e5)]:
        summary[name] = rank_candidates(df, scores)["id"].head(10).tolist()
    for technique, ids in qwen_results.items():  summary[f"Qwen-{technique}"]  = ids
    for technique, ids in glm_results.items():   summary[f"GLM-{technique}"]   = ids
    for technique, ids in gemma_results.items(): summary[f"Gemma-{technique}"] = ids

    tbl = pd.DataFrame(summary, index=range(1, 11))
    tbl.index.name = "Rank"
    tbl.to_csv("candidate_table.csv")
    files.download("candidate_table.csv")
    return tbl


# tbl = main()  # uncomment to run everything in one shot
